# APIs and Python

## What you will learn in this course 🧐🧐

APIs are one of the fastest ways to get data when it does not live in a file yet. You will learn how to:

- Explain what a REST API is and how the HTTP request and response cycle works
- Use `requests` to make GET and POST calls and read the response safely
- Parse JSON responses and pass query parameters
- Handle common failure modes like status codes and rate limits
- Build a reusable `fetch_prices()` function you can drop into a standalone script

<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D01-finpulse-logo.png"
  alt="FinPulse logo"
/>

You work as a junior data engineer at FinPulse, a small financial research team that tracks cryptocurrency markets.

The team needs fresh prices when a report is due, but right now someone pulls them by hand. Your manager wants an automated collector that fetches prices from an API and produces clean data that the rest of the team can reuse.

## What Is an API?

When data already sits in a local file, you can load it with pandas or another table-first tool. An API is what you use when the data lives on a remote service, and you have to ask a server for it.

A concrete example is flight search. A site like Kayak shows you prices because it asks airline systems for available flights and fares.

<Note type="definition" title="API">
An API, short for Application Programming Interface, is a set of rules that lets two software systems exchange data.
</Note>

The diagram below shows the same idea in one picture. A client sends a request through an API, and the server returns data in a response.

<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03D01-api_diagram.png"
  alt="Client app API layer and server showing request and response flow"
/>


## The HTTP Request and Response Cycle

Every time you call a web API, you send an HTTP request and you receive an HTTP response. The response has a status code that tells you if the call succeeded.

### Request

![HTTP request and response overview](https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D01-api-request.png)

When you read this diagram, focus on the parts you control in Python:

- The **method** (`GET`, `POST`, etc.)
- The **URL** (domain + path)
- Optional **query parameters** (the `params={...}` dict you pass to `requests`)
- Optional **headers** (auth tokens, content type, user agent)
- Optional **body** (common with `POST`, `PUT`, `PATCH`)

### Response

![HTTP response overview](https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D01-api-response.png)

When you read the response diagram, focus on what your code should check and parse:

- The **status code** (success vs client error vs server error)
- The **headers** (sometimes includes `Retry-After` for rate limits)
- The **body** (text, JSON, or bytes)


## Making GET Requests with `requests`

`requests` is the go-to Python library for calling HTTP endpoints. You will start with a tiny request to see what a response object looks like.

In [2]:
import time

import requests


This call hits a GitHub endpoint that returns a short sentence. The goal is to see the response object and confirm it succeeded.

In [3]:
r = requests.get("https://api.github.com/zen")
r

<Response [200]>

You mainly care about two things at first: the **status code** and the **body**.

- A `200` means success.
- A `4xx` means your request is wrong (bad URL, missing auth, invalid parameters).
- A `5xx` means the server had a problem.

Now read the body as text.

In [4]:
print("status:", r.status_code)
print("text:", r.text)

status: 200
text: Responsive is better than fast.


Some endpoints return JSON. GitHub’s API root is a simple example.

<Note type="warning" title="Text vs JSON">
A response can contain bytes (`r.content`), text (`r.text`), or JSON (`r.json()`). Only call `r.json()` when the response really is JSON.
</Note>

In [5]:
r_api = requests.get("https://api.github.com")
print("status:", r_api.status_code)
r_api.json()

status: 200


{'current_user_url': 'https://api.github.com/user',
 'current_user_authorizations_html_url': 'https://github.com/settings/connections/applications{/client_id}',
 'authorizations_url': 'https://api.github.com/authorizations',
 'code_search_url': 'https://api.github.com/search/code?q={query}{&page,per_page,sort,order}',
 'commit_search_url': 'https://api.github.com/search/commits?q={query}{&page,per_page,sort,order}',
 'emails_url': 'https://api.github.com/user/emails',
 'emojis_url': 'https://api.github.com/emojis',
 'events_url': 'https://api.github.com/events',
 'feeds_url': 'https://api.github.com/feeds',
 'followers_url': 'https://api.github.com/user/followers',
 'following_url': 'https://api.github.com/user/following{/target}',
 'gists_url': 'https://api.github.com/gists{/gist_id}',
 'hub_url': 'https://api.github.com/hub',
 'issue_search_url': 'https://api.github.com/search/issues?q={query}{&page,per_page,sort,order}',
 'issues_url': 'https://api.github.com/issues',
 'keys_url': '

## Status Codes and `raise_for_status()`

A status code can look “fine” at a glance, but your code should still fail fast when the API call failed. `raise_for_status()` turns a bad status into a Python exception.

In [ ]:
bad = requests.get("https://httpbin.org/status/404")
print("status:", bad.status_code)

try:
    bad.raise_for_status()
except requests.HTTPError as e:
    print("HTTPError (expected for this demo):", e)


status: 404
HTTPError (expected for this demo): 404 Client Error: NOT FOUND for url: https://httpbin.org/status/404


<Note type="tip" title="Make it a habit">
Call `raise_for_status()` before you parse the body. It stops your pipeline at the right place, and it prevents silent failures.
</Note>

## How query parameters fit into a URL

You will soon pass query parameters using `params={...}`. Visually, those parameters end up in the URL after a `?`.

![URI syntax diagram (scheme, authority, path, query, fragment)](https://upload.wikimedia.org/wikipedia/commons/d/d6/URI_syntax_diagram.svg)

This diagram is a **syntax map**, it shows which URL parts can appear, in which order. Read it left to right.

How to read it:

- Follow the main line, you can think of it as the "default" URL shape.
- Parts drawn as side branches are optional, many real URLs omit them.
- The punctuation characters are literal separators, like `://`, `/`, `?`, and `#`.

What the parts mean:

- **Scheme**: `https` or `http`
- **Authority**: the server part, usually a domain name like `api.github.com` (a port can also appear here, like `:443`)
- **Path**: the endpoint you want on that server, like `/repos/pallets/flask/issues`
- **Query**: the key/value pairs after `?`, like `?page=2&per_page=5`
- **Fragment**: the part after `#`, it is handled by the browser and is not sent to the server

A few concrete examples that match different branches in the diagram:

- Minimal, no path: `https://api.github.com`
- Path, no query: `https://api.github.com/zen`
- Path plus query parameters: `https://httpbin.org/get?q=finpulse&page=2`
- Local server with an explicit port: `http://localhost:8000/health`
- URL with a fragment (browser-side only): `https://site.example/docs/page#footnotes`

In `requests`, passing `params={"page": 2, "per_page": 5}` is how you build the **query** part without manually writing `?page=2&per_page=5`.

## Adding Parameters to Your Request

Most APIs take inputs through query parameters. `requests` lets you pass them as a Python dict, which avoids string-building bugs.

### Reading the CoinGecko API documentation

FinPulse pulls prices from **CoinGecko**. Before you call any endpoint, open their official API documentation so you build against what they support, not a URL you memorised by accident.

Use the CoinGecko API documentation link in **Resources** at the end of this notebook. There you will see the **base URL** and version path (`https://api.coingecko.com/api/v3/...`), each **endpoint** with its path, and the **query parameters** the server expects. For the simple price example below, the docs list `ids` (coin slugs such as `bitcoin`) and `vs_currencies` (such as `usd`). Check the **rate limit** section too, so you recognise why a `429` can appear and why spacing out calls matters in a busy classroom.

Keep that documentation open in a browser tab while you code. When something breaks or the response shape looks wrong, compare your request to the doc page before you change random parts of the URL.

Start with a neutral echo endpoint, then call the CoinGecko simple price endpoint.


In [6]:
echo = requests.get("https://httpbin.org/get", params={"q": "finpulse", "page": 2})
echo.raise_for_status()
echo.json()["args"]

{'page': '2', 'q': 'finpulse'}

Notice how the dictionary became a query string in the request. That is the core idea behind API inputs.

Now call CoinGecko. This endpoint is key-free and returns a small JSON object you can inspect easily.

In [7]:
coingecko_url = "https://api.coingecko.com/api/v3/simple/price"
params = {"ids": "bitcoin", "vs_currencies": "usd"}

cg = requests.get(coingecko_url, params=params, timeout=30)
if cg.status_code == 429:
    time.sleep(int(cg.headers.get("Retry-After", "30")))
    cg = requests.get(coingecko_url, params=params, timeout=30)
cg.raise_for_status()
time.sleep(0.35)
cg.json()


{'bitcoin': {'usd': 80660}}

## POST Requests

Many APIs let you create resources using POST. You will not build a full CRUD app in this notebook, but you should know the vocabulary.

This example sends form data to httpbin, which echoes it back.

In [8]:
payload = {"coin": "bitcoin", "action": "watch"}
posted = requests.post("https://httpbin.org/post", data=payload)
posted.raise_for_status()
posted.json()["form"]

{'action': 'watch', 'coin': 'bitcoin'}

## API Keys and Environment Variables

Some APIs require an API key. The safe pattern is to keep the key in an environment variable, then read it from Python.

<Note type="warning" title="Never commit API keys">
Do not put keys in a notebook, a script, or a `.env` file that is tracked by git. Even if you delete it later, git history can keep it.
</Note>

<Note type="important">
This is the exact pattern you will reuse any time you work with paid APIs.
</Note>

In [9]:
import os

api_key = os.getenv("MY_API_KEY")
if api_key is None:
    print("MY_API_KEY is not set. That is fine for this demo.")
else:
    print("MY_API_KEY is set. Keep it private.")

MY_API_KEY is not set. That is fine for this demo.


## Handling Rate Limits

Public APIs protect themselves. A common signal is status code `429`, which means you are calling too fast.

When the server sends a `Retry-After` header, you should wait that many seconds before trying again. Even when you are not rate-limited, adding a small delay in loops is a good habit.

<Note type="tip" title="Polite delay">
A small sleep, like 0.2 to 0.5 seconds between requests, reduces the chance of hitting rate limits and makes your script a better citizen.
</Note>

In [10]:
import time

def handle_rate_limit(status_code: int, headers: dict) -> int:
    """Return seconds to wait when rate-limited, else 0."""
    if status_code != 429:
        return 0

    retry_after = headers.get("Retry-After")
    wait_s = int(retry_after) if retry_after is not None else 5
    print(f"Rate limited (429). Waiting {wait_s}s before retrying.")
    return wait_s

wait_s = handle_rate_limit(429, {"Retry-After": "3"})
time.sleep(0.1)
wait_s

Rate limited (429). Waiting 3s before retrying.


3

## Paginating Through Results

Many endpoints return results page by page. The pattern is always the same: request page 1, then page 2, and stop when there is no more data.

You will see two common patterns:

- A loop that stops when the API returns an empty page
- A loop that uses a total count and computes the number of pages with `math.ceil`

In [11]:
from typing import Any


def fetch_github_issues(repo: str, per_page: int = 5, max_pages: int = 3) -> list[dict[str, Any]]:
    """Fetch a few pages of issues for a public repo (demo pagination)."""
    all_items: list[dict[str, Any]] = []

    for page in range(1, max_pages + 1):
        url = f"https://api.github.com/repos/{repo}/issues"
        r = requests.get(url, params={"per_page": per_page, "page": page, "state": "open"})
        r.raise_for_status()
        items = r.json()

        if not items:
            break

        all_items.extend(items)
        time.sleep(0.2)

    return all_items

issues = fetch_github_issues("pallets/flask", per_page=5, max_pages=2)
len(issues), issues[0]["title"]

(3, 'automatic options as separate route')

In [12]:
import math

total_items = 53
per_page = 10
pages = math.ceil(total_items / per_page)

pages

6

## Fetching Live Crypto Prices

The goal here is a small function that takes a list of coin ids, calls CoinGecko, and returns a clean Python dict. You can copy the finished function into any script that needs live prices.

Start with one request, then make it safer, then wrap it into a function.

In [13]:
coin_ids = ["bitcoin", "ethereum", "solana"]

url = "https://api.coingecko.com/api/v3/simple/price"
params = {"ids": ",".join(coin_ids), "vs_currencies": "usd"}

raw_prices = None
for attempt in range(5):
    cg = requests.get(url, params=params, timeout=30)
    if cg.status_code == 429:
        wait_s = int(cg.headers.get("Retry-After", "30"))
        print(f"429 from CoinGecko. Sleeping {wait_s}s (attempt {attempt + 1}/5)")
        time.sleep(wait_s)
        continue
    cg.raise_for_status()
    time.sleep(0.35)
    raw_prices = cg.json()
    break

raw_prices


{'bitcoin': {'usd': 80656},
 'ethereum': {'usd': 2369.53},
 'solana': {'usd': 84.62}}

In [14]:
def fetch_prices(coin_ids: list[str], vs_currency: str = "usd") -> dict:
    """Fetch current prices for a list of CoinGecko coin ids (with 429 retries)."""
    if not coin_ids:
        return {}

    url = "https://api.coingecko.com/api/v3/simple/price"
    params = {"ids": ",".join(coin_ids), "vs_currencies": vs_currency}

    for attempt in range(5):
        r = requests.get(url, params=params, timeout=30)
        if r.status_code == 429:
            wait_s = int(r.headers.get("Retry-After", "30"))
            print(f"429 from CoinGecko. Sleeping {wait_s}s (attempt {attempt + 1}/5)")
            time.sleep(wait_s)
            continue
        r.raise_for_status()
        time.sleep(0.35)
        return r.json()

    raise RuntimeError("CoinGecko still rate-limited after retries")


In [15]:
# One-shot call uses the same function as below (avoids duplicate CoinGecko patterns).
fetch_prices(["bitcoin"], "usd")


{'bitcoin': {'usd': 80660}}

In [ ]:
fetch_prices(["bitcoin", "ethereum", "solana"], "usd")

429 from CoinGecko. Sleeping 60s (attempt 1/5)


{'bitcoin': {'usd': 80661},
 'ethereum': {'usd': 2369.7},
 'solana': {'usd': 84.61}}

## Resources 📚📚

- [Requests quickstart](https://docs.python-requests.org/en/stable/user/quickstart/)
- [Public APIs](https://github.com/public-apis/public-apis)
- [CoinGecko API documentation](https://www.coingecko.com/en/api/documentation)
- [MDN HTTP overview](https://developer.mozilla.org/en-US/docs/Web/HTTP/Overview)
- [MDN HTTP response status codes](https://developer.mozilla.org/en-US/docs/Web/HTTP/Status)
- [URI syntax diagram (Wikimedia Commons)](https://commons.wikimedia.org/wiki/File:URI_syntax_diagram.svg)